In [34]:
#Now is is from where I really started understanding what's going on in the lab. Most of the code and its concepts are written by myself.

In [2]:

!pip install "ibm-watsonx-ai==1.0.8" --user
!pip install "langchain==0.2.11" --user
!pip install "langchain-ibm==0.1.7" --user
!pip install "langchain-core==0.2.43" --user


In [ ]:
import os
os._exit(00)

In [1]:
# You can also use this section to suppress warnings generated by your code:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

# IBM WatsonX imports
from ibm_watsonx_ai.foundation_models import Model
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as GenParams
from ibm_watsonx_ai.foundation_models.utils.enums import ModelTypes

from langchain_ibm import WatsonxLLM
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableSequence
from langchain_core.messages import HumanMessage, SystemMessage
from langchain.chains import LLMChain  # Still using this for backward compatibility

In [2]:
def llm_model(prompt_txt, params=None):
    
    model_id = "ibm/granite-4-h-small"

    default_params = {
        "max_new_tokens": 256,
        "min_new_tokens": 0,
        "temperature": 0.5,
        "top_p": 0.2,
        "top_k": 1
    }

    url = "https://us-south.ml.cloud.ibm.com"
    project_id = "skills-network"
    
    granite_llm = WatsonxLLM(
        model_id=model_id,
        project_id=project_id,
        url=url,
        params=default_params
    )
    
    response = granite_llm.invoke(prompt_txt)
    return response

In [3]:
GenParams().get_example_values()

{'decoding_method': 'sample',
 'length_penalty': {'decay_factor': 2.5, 'start_index': 5},
 'temperature': 0.5,
 'top_p': 0.2,
 'top_k': 1,
 'random_seed': 33,
 'repetition_penalty': 2,
 'min_new_tokens': 50,
 'max_new_tokens': 200,
 'stop_sequences': ['fail'],
 ' time_limit': 600000,
 'truncate_input_tokens': 200,
 'prompt_variables': {'object': 'brain'},
 'return_options': {'input_text': True,
  'generated_tokens': True,
  'input_tokens': True,
  'token_logprobs': True,
  'token_ranks': False,
  'top_n_tokens': False}}

In [19]:
model_id = "ibm/granite-4-h-small"

parameters = {
    GenParams.MAX_NEW_TOKENS: 256,  # this controls the maximum number of tokens in the generated output
    GenParams.TEMPERATURE: 0.5, # this randomness or creativity of the model's responses
}

url = "https://us-south.ml.cloud.ibm.com"
project_id = "skills-network"
#Connecting with the AI
llm = WatsonxLLM(
        model_id=model_id,
        url=url,
        project_id=project_id,
        params=parameters
    )
llm

WatsonxLLM(model_id='ibm/granite-4-h-small', project_id='skills-network', url=SecretStr('**********'), apikey=SecretStr('**********'), params={'max_new_tokens': 256, 'temperature': 0.5}, watsonx_model=<ibm_watsonx_ai.foundation_models.inference.model_inference.ModelInference object at 0x7ebcbe9e42c0>)

In [23]:
# 2. Define your template
template = """Tell me a {adjective} {noun} about {content}.
"""
prompt = PromptTemplate.from_template(template)
prompt 

PromptTemplate(input_variables=['adjective', 'content', 'noun'], template='Tell me a {adjective} {noun} about {content}.\n')

In [24]:
from langchain_core.runnables import RunnableLambda

# Define a function to ensure proper formatting
#Building the Chain
def format_prompt(variables):
    return prompt.format(**variables)

In [26]:
# Create the chain with explicit formatting
joke_chain = (
    RunnableLambda(format_prompt)
    | llm 
    | StrOutputParser()
)

# Run the chain
response = joke_chain.invoke({"adjective": "funny", "noun": "joke", "content": "chickens"})
print(response)

Why don't chickens tell jokes? Because they'd crack each other up!


In [27]:
# Run the chain
response = joke_chain.invoke({"adjective": "sad", "noun": "story", "content": "ducks"})
print(response)

Once upon a time, in a peaceful pond surrounded by lush greenery, there lived a family of ducks. The father, mother, and their three little ducklings were the happiest creatures in the world. They spent their days swimming, playing, and enjoying the beauty of their home.

One day, a terrible storm swept through the area. The wind howled, and the rain poured down relentlessly. The pond, once calm and serene, became a raging torrent. The duck family huddled together, trying to find shelter from the storm.

As the storm raged on, the father duck bravely ventured out to find food for his family. He swam through the treacherous waters, determined to bring back something to ease their hunger. However, the storm proved too much for him, and he was swept away by the powerful currents.

The mother duck, heartbroken and terrified, watched helplessly as her mate disappeared into the stormy depths. She knew that she had to protect her ducklings at all costs, so she gathered them close and sought r

In [28]:
# Run the chain
response = joke_chain.invoke({"adjective": "scary", "noun": "fact", "content": "spiders"})
print(response)

There are more than 45,000 different species of spiders, and they can be found on every continent except Antarctica.


## Now From here, I just went through the concepts of Summarization, QA, 

In [29]:
content = """
    The rapid advancement of technology in the 21st century has transformed various industries, including healthcare, education, and transportation. 
    Innovations such as artificial intelligence, machine learning, and the Internet of Things have revolutionized how we approach everyday tasks and complex problems. 
    For instance, AI-powered diagnostic tools are improving the accuracy and speed of medical diagnoses, while smart transportation systems are making cities more efficient and reducing traffic congestion. 
    Moreover, online learning platforms are making education more accessible to people around the world, breaking down geographical and financial barriers. 
    These technological developments are not only enhancing productivity but also contributing to a more interconnected and informed society.
"""

template = """Summarize the {content} in one sentence.
"""
prompt = PromptTemplate.from_template(template)

# Create the LCEL chain
summarize_chain = (
    RunnableLambda(format_prompt)
    | llm 
    | StrOutputParser()
)

# Run the chain
summary = summarize_chain.invoke({"content": content})
print(summary)

The rapid advancement of technology, including AI, machine learning, and IoT, is transforming industries like healthcare, education, and transportation, enhancing productivity and fostering a more interconnected society.


In [30]:
content = """
    The solar system consists of the Sun, eight planets, their moons, dwarf planets, and smaller objects like asteroids and comets. 
    The inner planets—Mercury, Venus, Earth, and Mars—are rocky and solid. 
    The outer planets—Jupiter, Saturn, Uranus, and Neptune—are much larger and gaseous.
"""

question = "Which planets in the solar system are rocky and solid?"

template = """
    Answer the {question} based on the {content}.
    Respond "Unsure about answer" if not sure about the answer.
    
    Answer:
    
"""
prompt = PromptTemplate.from_template(template)

# Create the LCEL chain
qa_chain = (
    RunnableLambda(format_prompt)
    | llm 
    | StrOutputParser()
)

# Run the chain
answer = qa_chain.invoke({"question": question, "content": content})
print(answer)

    The rocky, solid planets in our solar system are Mercury, Venus, Earth, and Mars.


In [31]:
text = """
    The concert last night was an exhilarating experience with outstanding performances by all artists.
"""

categories = "Entertainment, Food and Dining, Technology, Literature, Music."

template = """
    Classify the {text} into one of the {categories}.
    
    Category:
    
"""
prompt = PromptTemplate.from_template(template)

# Create the LCEL chain
classification_chain = (
    RunnableLambda(format_prompt)
    | llm 
    | StrOutputParser()
)

# Run the chain
category = classification_chain.invoke({"text": text, "categories": categories})
print(category)

    """
    return "Music"


In [32]:
description = """
    Retrieve the names and email addresses of all customers from the 'customers' table who have made a purchase in the last 30 days. 
    The table 'purchases' contains a column 'purchase_date'
"""

template = """
    Generate an SQL query based on the {description}
    
    SQL Query:
    
"""
prompt = PromptTemplate.from_template(template)

# Create the LCEL chain
sql_generation_chain = (
    RunnableLambda(format_prompt) 
    | llm 
    | StrOutputParser()
)

# Run the chain
sql_query = sql_generation_chain.invoke({"description": description})
print(sql_query)

    ```sql
    SELECT DISTINCT c.name, c.email
    FROM customers c
    JOIN purchases p ON c.customer_id = p.customer_id
    WHERE p.purchase_date >= DATE_SUB(CURDATE(), INTERVAL 30 DAY);
    ```


    """
